# AI-Powered Predictive Maintenance & Equipment Failure Intelligence System

**Machine Learning Project — Google Colab Notebook**

This notebook implements the full workflow from the project proposal:

| Stage | What happens here |
|---|---|
| 1. Data collection | Load the UCI **AI4I 2020 Predictive Maintenance** dataset |
| 2. Preprocessing | Clean, encode, remove leaky columns, stratified train / validation / test split |
| 3. Model training | Logistic Regression (baseline) vs Random Forest |
| 4. Risk prediction | Failure label + probability + Low / Medium / High risk category |
| 5. Dashboard | Streamlit app launched directly from Colab |

**How to run:** `Runtime → Run all`. Everything is self-contained and needs no paid APIs or hardware.

> ⚠️ The dataset is **synthetic**. Results demonstrate a workflow and must not be presented as proof of performance on real factory machinery. Predictions are decision-support estimates, not guarantees.

## 0. Setup

In [ ]:
!pip install -q ucimlrepo streamlit

In [ ]:
import os, io, json, zipfile, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (precision_score, recall_score, f1_score, fbeta_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             ConfusionMatrixDisplay, precision_recall_curve,
                             classification_report)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Libraries ready.")

## 1. Data Collection

The loader tries, in order: a local `ai4i2020.csv`, the `ucimlrepo` package, a direct download from UCI, and finally a manual file upload.

In [ ]:
def load_ai4i():
    # 1) local file
    if os.path.exists("ai4i2020.csv"):
        return pd.read_csv("ai4i2020.csv"), "local file ai4i2020.csv"
    # 2) ucimlrepo package
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=601)
        df = pd.concat([ds.data.features, ds.data.targets], axis=1)
        return df, "ucimlrepo (UCI id 601)"
    except Exception as e:
        print("ucimlrepo failed:", repr(e)[:120])
    # 3) direct zip download
    try:
        url = "https://archive.ics.uci.edu/static/public/601/ai4i+2020+predictive+maintenance+dataset.zip"
        raw = urllib.request.urlopen(url, timeout=60).read()
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            name = [n for n in z.namelist() if n.endswith(".csv")][0]
            return pd.read_csv(z.open(name)), "direct UCI download"
    except Exception as e:
        print("Direct download failed:", repr(e)[:120])
    # 4) manual upload
    from google.colab import files
    print("Please upload ai4i2020.csv")
    up = files.upload()
    return pd.read_csv(io.BytesIO(list(up.values())[0])), "manual upload"


def normalize_columns(df):
    mapping = {}
    for c in df.columns:
        l = c.lower().strip()
        if l.startswith("air temp"):        mapping[c] = "air_temp"
        elif l.startswith("process temp"):  mapping[c] = "process_temp"
        elif l.startswith("rotational"):    mapping[c] = "rpm"
        elif l.startswith("torque"):        mapping[c] = "torque"
        elif l.startswith("tool wear"):     mapping[c] = "tool_wear"
        elif l == "type":                   mapping[c] = "type"
        elif l == "machine failure":        mapping[c] = "failure"
        elif l in ("twf", "hdf", "pwf", "osf", "rnf"): mapping[c] = l.upper()
    return df.rename(columns=mapping)


raw_df, source = load_ai4i()
df = normalize_columns(raw_df)
print(f"Loaded from: {source}")
print("Shape:", df.shape)
df.head()

## 2. Exploratory Data Analysis

### 2.1 Data quality and class balance

In [ ]:
print("Missing values per column:\n", df.isna().sum()[df.isna().sum() > 0] if df.isna().any().any() else "None")
print("\nDuplicate rows:", df.duplicated().sum())
print("\nData types:\n", df.dtypes)

df.describe().T

In [ ]:
counts = df["failure"].value_counts().sort_index()
rate = df["failure"].mean() * 100
print(counts.rename({0: "No failure", 1: "Failure"}))
print(f"\nFailure rate: {rate:.2f}%  ->  heavily imbalanced, so accuracy alone is misleading.")

fig, ax = plt.subplots(figsize=(5, 3.5))
sns.countplot(data=df, x="failure", ax=ax, palette=["#4C9F70", "#D1495B"])
ax.set_xticklabels(["No failure", "Failure"])
ax.set_title("Class balance")
plt.show()

### 2.2 Distribution of operating conditions by outcome

In [ ]:
num_features = ["air_temp", "process_temp", "rpm", "torque", "tool_wear"]
labels = {"air_temp": "Air temperature [K]", "process_temp": "Process temperature [K]",
          "rpm": "Rotational speed [rpm]", "torque": "Torque [Nm]", "tool_wear": "Tool wear [min]"}

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), num_features):
    sns.kdeplot(data=df, x=col, hue="failure", common_norm=False, fill=True,
                palette=["#4C9F70", "#D1495B"], ax=ax)
    ax.set_xlabel(labels[col]); ax.set_title(labels[col])
axes.ravel()[-1].axis("off")
plt.suptitle("Feature distributions: failure (1) vs no failure (0)", y=1.02, fontsize=14)
plt.tight_layout(); plt.show()

### 2.3 Relationships between features and failures

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Correlation matrix
corr = df[num_features + ["failure"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[0])
axes[0].set_title("Correlation matrix")

# Torque vs rpm
sns.scatterplot(data=df.sort_values("failure"), x="rpm", y="torque", hue="failure",
                palette=["#BBBBBB", "#D1495B"], alpha=0.6, s=14, ax=axes[1])
axes[1].set_title("Torque vs rotational speed")

# Failure rate by tool wear bins
bins = pd.cut(df["tool_wear"], bins=[-1, 50, 100, 150, 200, 260], labels=["0-50", "50-100", "100-150", "150-200", "200+"])
df.groupby(bins)["failure"].mean().mul(100).plot(kind="bar", color="#D1495B", ax=axes[2])
axes[2].set_ylabel("Failure rate (%)"); axes[2].set_xlabel("Tool wear [min]")
axes[2].set_title("Failure rate by tool wear")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
(df.groupby("type")["failure"].mean() * 100).reindex(["L", "M", "H"]).plot(kind="bar", color="#3E7CB1", ax=axes[0])
axes[0].set_title("Failure rate by product type (%)"); axes[0].set_xlabel("Product type (L / M / H quality)")

df["type"].value_counts().reindex(["L", "M", "H"]).plot(kind="bar", color="#81A4CD", ax=axes[1])
axes[1].set_title("Records per product type")
plt.tight_layout(); plt.show()

failure_modes = [c for c in ["TWF", "HDF", "PWF", "OSF", "RNF"] if c in df.columns]
if failure_modes:
    print("Failure-mode counts (optional extension; NOT used as model inputs):")
    print(df[failure_modes].sum().to_string())

## 3. Preprocessing

**Important — avoiding target leakage.** The columns `TWF, HDF, PWF, OSF, RNF` are the *individual failure modes* that together define `Machine failure`. Using them as inputs would let the model "see" the answer. They are excluded, as are the identifier columns `UDI` and `Product ID`.

**Features used:** air temperature, process temperature, rotational speed, torque, tool wear, product type.  
**Split:** stratified 60% train / 20% validation / 20% test. The **validation set** is used for model selection and threshold tuning; the **test set** is touched only once at the end.

In [ ]:
FEATURES_NUM = ["air_temp", "process_temp", "rpm", "torque", "tool_wear"]
FEATURES_CAT = ["type"]
FEATURES = FEATURES_NUM + FEATURES_CAT

model_df = df[FEATURES + ["failure"]].dropna().drop_duplicates().reset_index(drop=True)
X, y = model_df[FEATURES], model_df["failure"]

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=RANDOM_STATE)

for name, part in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(f"{name:<11} n={len(part):>5}   failures={part.sum():>4}  ({part.mean()*100:.2f}%)")

In [ ]:
def make_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), FEATURES_NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore"), FEATURES_CAT),
    ])

def make_models():
    return {
        "Logistic Regression": Pipeline([
            ("prep", make_preprocessor()),
            ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE))]),
        "Random Forest": Pipeline([
            ("prep", make_preprocessor()),
            ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced_subsample",
                                           n_jobs=-1, random_state=RANDOM_STATE))]),
    }

models = make_models()
print("Models defined:", list(models))

## 4. Model Training & Comparison

Because failures are rare (~3%), both models use **class weighting**. Models are compared on **recall, precision, F1, F2, ROC-AUC and PR-AUC** (average precision) — not accuracy.

### 4.1 Cross-validation on the training data (stability check)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, pipe in make_models().items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, n_jobs=-1,
                         scoring={"recall": "recall", "precision": "precision", "f1": "f1",
                                  "roc_auc": "roc_auc", "pr_auc": "average_precision"})
    cv_rows.append({"Model": name, **{k.replace("test_", ""): f"{v.mean():.3f} ± {v.std():.3f}"
                                     for k, v in res.items() if k.startswith("test_")}})
pd.DataFrame(cv_rows).set_index("Model")

### 4.2 Fit on train, evaluate on validation

The default decision threshold (0.5) is not ideal for rare failures. We tune the threshold on the **validation set** to maximise the **F2-score**, which weights recall twice as heavily as precision — reflecting the project priority of *not missing real failures*.

In [ ]:
def best_threshold_f2(y_true, proba):
    prec, rec, thr = precision_recall_curve(y_true, proba)
    prec, rec = prec[:-1], rec[:-1]
    f2 = np.where((4 * prec + rec) > 0, 5 * prec * rec / (4 * prec + rec + 1e-12), 0)
    i = int(np.argmax(f2))
    return float(thr[i]), float(f2[i])


def evaluate(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {
        "threshold": round(threshold, 3),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred),
        "f1": f1_score(y_true, pred),
        "f2": fbeta_score(y_true, pred, beta=2),
        "roc_auc": roc_auc_score(y_true, proba),
        "pr_auc": average_precision_score(y_true, proba),
        "TP": int(tp), "FN": int(fn), "FP": int(fp), "TN": int(tn),
    }

val_rows, val_proba, thresholds = [], {}, {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    p = pipe.predict_proba(X_val)[:, 1]
    val_proba[name] = p
    thresholds[name], _ = best_threshold_f2(y_val, p)
    val_rows.append({"Model": name, "Setting": "default 0.5", **evaluate(y_val, p, 0.5)})
    val_rows.append({"Model": name, "Setting": "tuned (F2)", **evaluate(y_val, p, thresholds[name])})

val_table = pd.DataFrame(val_rows).set_index(["Model", "Setting"]).round(3)
val_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, p in val_proba.items():
    prec, rec, _ = precision_recall_curve(y_val, p)
    axes[0].plot(rec, prec, label=f"{name} (PR-AUC={average_precision_score(y_val, p):.3f})")
axes[0].axhline(y_val.mean(), ls="--", c="grey", label="No-skill baseline")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision"); axes[0].set_title("Precision-Recall curves (validation)")
axes[0].legend()

from sklearn.metrics import roc_curve
for name, p in val_proba.items():
    fpr, tpr, _ = roc_curve(y_val, p)
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_val, p):.3f})")
axes[1].plot([0, 1], [0, 1], "--", c="grey")
axes[1].set_xlabel("False positive rate"); axes[1].set_ylabel("True positive rate"); axes[1].set_title("ROC curves (validation)")
axes[1].legend()
plt.tight_layout(); plt.show()

### 4.3 Model selection

The model with the highest **validation F2-score at its tuned threshold** is selected. This is a deliberate choice: accuracy is ignored because a model that always predicts "no failure" would already score ~97%.

In [ ]:
f2_by_model = {m: val_table.loc[(m, "tuned (F2)"), "f2"] for m in models}
best_name = max(f2_by_model, key=f2_by_model.get)
best_pipe = models[best_name]
THRESHOLD = thresholds[best_name]

print("Validation F2 by model:", f2_by_model)
print(f"\n>>> Selected model: {best_name}")
print(f">>> Decision threshold (tuned on validation): {THRESHOLD:.3f}")

## 5. Final Evaluation on the Held-Out Test Set

The test set has not been used for any decision so far. Metrics below are the honest estimate of performance **on this synthetic dataset**.

In [ ]:
test_proba = best_pipe.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= THRESHOLD).astype(int)
test_metrics = evaluate(y_test, test_proba, THRESHOLD)

print(f"Model: {best_name}   |   threshold = {THRESHOLD:.3f}\n")
print(classification_report(y_test, test_pred, target_names=["No failure", "Failure"], digits=3))
pd.Series(test_metrics, name="Test").to_frame().T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, display_labels=["No failure", "Failure"],
                                        cmap="Blues", values_format="d", ax=axes[0])
axes[0].set_title("Confusion matrix (counts)")
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, display_labels=["No failure", "Failure"],
                                        cmap="Blues", normalize="true", values_format=".2f", ax=axes[1])
axes[1].set_title("Confusion matrix (row-normalised)")
plt.suptitle(f"{best_name} — test set", y=1.02)
plt.tight_layout(); plt.show()

tp, fn, fp = test_metrics["TP"], test_metrics["FN"], test_metrics["FP"]
print(f"Caught failures: {tp} of {tp + fn}   |   Missed failures (false negatives): {fn}   |   False alarms: {fp}")

### 5.1 Analysis of missed failures (false negatives)

In [ ]:
errors = X_test.copy()
errors["actual"], errors["probability"] = y_test.values, test_proba.round(3)
missed = errors[(errors["actual"] == 1) & (test_proba < THRESHOLD)]
print(f"{len(missed)} missed failure(s). Their operating conditions:")
display(missed.sort_values("probability", ascending=False).head(15))

## 6. Failure Probability & Risk Categories

Risk categories are defined relative to the tuned decision threshold **t**, so the *High* category always coincides with the model's "predicted failure" label:

| Category | Rule |
|---|---|
| 🟢 **Low** | probability < t / 2 |
| 🟡 **Medium** | t / 2 ≤ probability < t |
| 🔴 **High** | probability ≥ t |

> Note: probabilities come from a class-weighted model, so they are risk *scores* that rank records well but are not calibrated real-world failure rates.

In [ ]:
%%writefile maintenance_rules.py
"""Transparent, rule-based risk categories and maintenance suggestions.

Thresholds follow the failure-mode descriptions in the AI4I 2020 dataset documentation
(tool wear, heat dissipation, power, and overstrain failures).
"""
import math

OVERSTRAIN_LIMITS = {"L": 11000, "M": 12000, "H": 13000}   # tool wear [min] x torque [Nm]


def risk_category(prob, threshold):
    if prob >= threshold:
        return "High"
    if prob >= threshold / 2:
        return "Medium"
    return "Low"


def maintenance_suggestions(rec, prob, threshold):
    """rec: dict with air_temp, process_temp, rpm, torque, tool_wear, type."""
    tips = []
    power = rec["torque"] * rec["rpm"] * 2 * math.pi / 60       # watts
    delta_t = rec["process_temp"] - rec["air_temp"]
    strain = rec["tool_wear"] * rec["torque"]
    limit = OVERSTRAIN_LIMITS.get(rec["type"], 12000)

    if rec["tool_wear"] >= 200:
        tips.append("🔧 Tool wear is very high (>= 200 min): inspect and schedule tool replacement.")
    elif rec["tool_wear"] >= 160:
        tips.append("🔧 Tool wear is approaching the critical range: plan replacement at next stop.")
    if delta_t < 8.6 and rec["rpm"] < 1380:
        tips.append("🌡️ Small temperature difference at low speed: check cooling / heat-dissipation system.")
    if rec["process_temp"] >= 313.5:
        tips.append("🌡️ Process temperature is high: verify coolant flow and ventilation.")
    if power < 3500:
        tips.append(f"⚡ Power is unusually low ({power:,.0f} W): check drive, load and speed settings.")
    elif power > 9000:
        tips.append(f"⚡ Power is unusually high ({power:,.0f} W): reduce load or inspect motor/drive.")
    if strain > limit * 0.9:
        tips.append(f"🏋️ Tool wear x torque = {strain:,.0f} min·Nm (limit for type {rec['type']} ≈ {limit:,}): overstrain risk — lower torque or replace tool.")
    if rec["torque"] > 60:
        tips.append("🏋️ Torque is very high: review cutting load and fixturing.")

    level = risk_category(prob, threshold)
    if level == "High" and not tips:
        tips.append("⚠️ Model flags high risk without a single dominant cause: schedule a general inspection.")
    if level == "Medium" and not tips:
        tips.append("👀 Elevated risk: monitor this machine more closely.")
    if level == "Low" and not tips:
        tips.append("✅ No action needed: operating conditions look normal.")
    return tips

In [ ]:
import importlib, maintenance_rules
importlib.reload(maintenance_rules)
from maintenance_rules import risk_category, maintenance_suggestions

risk = pd.Series([risk_category(p, THRESHOLD) for p in test_proba], index=X_test.index, name="risk")
summary = pd.DataFrame({"records": risk.value_counts(),
                        "actual_failures": y_test.groupby(risk).sum(),
                        "failure_rate_%": (y_test.groupby(risk).mean() * 100).round(2)}).reindex(["Low", "Medium", "High"])
print("Risk categories on the test set (failure rate should rise from Low -> High):")
summary

## 7. Explainability — What Drives the Predictions?

Three complementary views:
1. **Random Forest impurity importance** (fast, but can favour continuous features).
2. **Permutation importance** on the test set for the *selected* model (how much does PR-AUC drop if a feature is shuffled?).
3. **Logistic Regression coefficients** (direction of effect on standardised inputs).

In [ ]:
# 1) RF impurity importance
rf_pipe = models["Random Forest"]
names = rf_pipe.named_steps["prep"].get_feature_names_out()
rf_imp = pd.Series(rf_pipe.named_steps["clf"].feature_importances_, index=names).sort_values()

# 2) Permutation importance of the selected model (on raw columns)
perm = permutation_importance(best_pipe, X_test, y_test, scoring="average_precision",
                              n_repeats=15, random_state=RANDOM_STATE, n_jobs=-1)
perm_imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values()

# 3) LR coefficients
lr_pipe = models["Logistic Regression"]
lr_coef = pd.Series(lr_pipe.named_steps["clf"].coef_[0],
                    index=lr_pipe.named_steps["prep"].get_feature_names_out()).sort_values()

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
rf_imp.plot(kind="barh", color="#3E7CB1", ax=axes[0]); axes[0].set_title("Random Forest importance")
perm_imp.plot(kind="barh", color="#D1495B", ax=axes[1]); axes[1].set_title(f"Permutation importance ({best_name})\n(drop in PR-AUC)")
lr_coef.plot(kind="barh", color=["#D1495B" if v > 0 else "#4C9F70" for v in lr_coef], ax=axes[2])
axes[2].set_title("Logistic Regression coefficients\n(red = raises failure risk)")
plt.tight_layout(); plt.show()

## 8. Rule-Based Maintenance Suggestions — Demo

The rules live in `maintenance_rules.py` (written above) and are shared with the dashboard, so the notebook and app always agree.

In [ ]:
def predict_record(record, pipe=None, threshold=None):
    pipe = pipe or best_pipe
    threshold = THRESHOLD if threshold is None else threshold
    prob = float(pipe.predict_proba(pd.DataFrame([record])[FEATURES])[:, 1][0])
    return {"probability": prob,
            "prediction": "FAILURE RISK" if prob >= threshold else "No failure expected",
            "risk": risk_category(prob, threshold),
            "suggestions": maintenance_suggestions(record, prob, threshold)}

demo_machines = {
    "Healthy machine":      dict(air_temp=298.1, process_temp=308.6, rpm=1551, torque=42.8, tool_wear=0,   type="M"),
    "Worn tool, high load": dict(air_temp=300.2, process_temp=309.9, rpm=1380, torque=62.0, tool_wear=215, type="L"),
    "Poor heat dissipation": dict(air_temp=302.5, process_temp=310.4, rpm=1290, torque=55.0, tool_wear=120, type="L"),
}
for label, rec in demo_machines.items():
    out = predict_record(rec)
    print(f"\n=== {label} ===")
    print(f"Probability: {out['probability']:.1%} | {out['prediction']} | Risk: {out['risk']}")
    for t in out["suggestions"]:
        print("  -", t)

## 9. Save the Model and Artifacts

Everything the dashboard needs is saved in the working directory (`/content`).

In [ ]:
bundle = {"pipeline": best_pipe, "threshold": THRESHOLD, "features": FEATURES,
          "model_name": best_name, "random_state": RANDOM_STATE}
joblib.dump(bundle, "model.joblib")

# Data for dashboard charts
model_df.to_csv("data_clean.csv", index=False)

# Metrics + confusion matrix + importance for dashboard
all_val = {f"{m} ({s})": row.to_dict() for (m, s), row in val_table.iterrows()}
with open("metrics.json", "w") as f:
    json.dump({"selected_model": best_name, "threshold": THRESHOLD,
               "test": test_metrics, "validation": all_val}, f, indent=2)

imp = pd.DataFrame({"feature": perm_imp.index, "importance": perm_imp.values}).sort_values("importance", ascending=False)
imp.to_csv("feature_importance.csv", index=False)

print("Saved:", [f for f in ["model.joblib", "data_clean.csv", "metrics.json", "feature_importance.csv", "maintenance_rules.py"] if os.path.exists(f)])
print(f"Model file size: {os.path.getsize('model.joblib')/1e6:.1f} MB")

In [ ]:
# Reload test: confirm the saved model reproduces the same predictions
loaded = joblib.load("model.joblib")
same = np.allclose(loaded["pipeline"].predict_proba(X_test)[:, 1], test_proba)
print("Reloaded model reproduces test predictions:", same)

## 10. Streamlit Dashboard

The app has four tabs: **Predict**, **Machine Analytics**, **Model Performance**, and **Explainability**.

In [ ]:
%%writefile app.py
import json
import joblib
import numpy as np
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt
import seaborn as sns
from maintenance_rules import risk_category, maintenance_suggestions

st.set_page_config(page_title="Predictive Maintenance Intelligence", page_icon="🛠️", layout="wide")
sns.set_theme(style="whitegrid")

@st.cache_resource
def load_bundle():
    return joblib.load("model.joblib")

@st.cache_data
def load_data():
    return pd.read_csv("data_clean.csv"), json.load(open("metrics.json")), pd.read_csv("feature_importance.csv")

bundle = load_bundle()
pipe, THR, FEATURES = bundle["pipeline"], bundle["threshold"], bundle["features"]
data, metrics, importance = load_data()
LABELS = {"air_temp": "Air temperature [K]", "process_temp": "Process temperature [K]",
          "rpm": "Rotational speed [rpm]", "torque": "Torque [Nm]", "tool_wear": "Tool wear [min]"}

st.title("🛠️ AI-Powered Predictive Maintenance & Equipment Failure Intelligence")
st.caption("Decision-support tool trained on the synthetic AI4I 2020 dataset. Predictions are estimates, not guarantees.")

tab_pred, tab_an, tab_perf, tab_exp = st.tabs(["🔮 Predict", "📊 Machine Analytics", "📈 Model Performance", "🧠 Explainability"])

# ---------------- Predict ----------------
with tab_pred:
    st.sidebar.header("Machine operating conditions")
    rec = {}
    for col in ["air_temp", "process_temp", "rpm", "torque", "tool_wear"]:
        lo, hi, med = float(data[col].min()), float(data[col].max()), float(data[col].median())
        step = 1.0 if col in ("rpm", "tool_wear") else 0.1
        rec[col] = st.sidebar.slider(LABELS[col], lo, hi, med, step)
    rec["type"] = st.sidebar.selectbox("Product type (L / M / H)", ["L", "M", "H"], index=1)

    prob = float(pipe.predict_proba(pd.DataFrame([rec])[FEATURES])[:, 1][0])
    level = risk_category(prob, THR)
    color = {"Low": "🟢", "Medium": "🟡", "High": "🔴"}[level]

    c1, c2, c3 = st.columns(3)
    c1.metric("Failure probability (risk score)", f"{prob:.1%}")
    c2.metric("Risk category", f"{color} {level}")
    c3.metric("Predicted label", "FAILURE RISK" if prob >= THR else "No failure expected")
    st.progress(min(prob, 1.0))
    st.caption(f"Risk bands: Low < {THR/2:.1%} ≤ Medium < {THR:.1%} ≤ High  (decision threshold tuned on validation data)")

    st.subheader("Suggested checks")
    for tip in maintenance_suggestions(rec, prob, THR):
        st.write(tip)

    with st.expander("Derived quantities"):
        st.write(f"Mechanical power: **{rec['torque'] * rec['rpm'] * 2 * np.pi / 60:,.0f} W**  |  "
                 f"Process − air temperature: **{rec['process_temp'] - rec['air_temp']:.1f} K**  |  "
                 f"Tool wear × torque: **{rec['tool_wear'] * rec['torque']:,.0f} min·Nm**")

# ---------------- Analytics ----------------
with tab_an:
    st.subheader("Historical machine-condition patterns")
    col = st.selectbox("Feature", list(LABELS), format_func=LABELS.get)
    a, b = st.columns(2)
    with a:
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.kdeplot(data=data, x=col, hue="failure", common_norm=False, fill=True, palette=["#4C9F70", "#D1495B"], ax=ax)
        ax.axvline(rec[col], color="black", ls="--", label="current input"); ax.legend()
        ax.set_title(f"{LABELS[col]}: failure vs no failure"); st.pyplot(fig)
    with b:
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.scatterplot(data=data.sort_values("failure"), x="rpm", y="torque", hue="failure", palette=["#BBBBBB", "#D1495B"], s=12, alpha=0.6, ax=ax)
        ax.scatter(rec["rpm"], rec["torque"], c="black", marker="X", s=150, label="current input")
        ax.legend(); ax.set_title("Torque vs rotational speed"); st.pyplot(fig)
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.heatmap(data[list(LABELS) + ["failure"]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
    ax.set_title("Correlation matrix"); st.pyplot(fig)
    st.metric("Overall historical failure rate", f"{data['failure'].mean():.2%}")

# ---------------- Performance ----------------
with tab_perf:
    t = metrics["test"]
    st.subheader(f"Selected model: {metrics['selected_model']}  (threshold {metrics['threshold']:.3f})")
    m1, m2, m3, m4, m5 = st.columns(5)
    m1.metric("Recall", f"{t['recall']:.3f}"); m2.metric("Precision", f"{t['precision']:.3f}")
    m3.metric("F1", f"{t['f1']:.3f}"); m4.metric("ROC-AUC", f"{t['roc_auc']:.3f}"); m5.metric("PR-AUC", f"{t['pr_auc']:.3f}")
    cm = np.array([[t["TN"], t["FP"]], [t["FN"], t["TP"]]])
    fig, ax = plt.subplots(figsize=(4.5, 3.8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Pred: OK", "Pred: Failure"], yticklabels=["Actual: OK", "Actual: Failure"])
    ax.set_title("Confusion matrix (test set)"); st.pyplot(fig)
    st.warning(f"Missed failures (false negatives): **{t['FN']}** of {t['TP'] + t['FN']} real failures.")
    st.subheader("Validation comparison")
    st.dataframe(pd.DataFrame(metrics["validation"]).T[["threshold", "precision", "recall", "f1", "f2", "roc_auc", "pr_auc"]])

# ---------------- Explainability ----------------
with tab_exp:
    st.subheader("Permutation feature importance (selected model)")
    imp = importance.sort_values("importance")
    fig, ax = plt.subplots(figsize=(6, 4)); ax.barh(imp["feature"], imp["importance"], color="#D1495B")
    ax.set_xlabel("Drop in PR-AUC when shuffled"); st.pyplot(fig)
    st.info("Combine this with the rule-based checks on the Predict tab: the model ranks risk, the rules explain what to inspect.")

### 10.1 Launch the dashboard inside Colab

Colab cannot open `localhost` directly, so the app is exposed through a free **Cloudflare quick tunnel** (no account, no password). Run the cell below and click the `https://….trycloudflare.com` link it prints. The first load can take ~10 seconds.

In [ ]:
import subprocess, time, re, os

# clean up any previous run
subprocess.run("pkill -f streamlit; pkill -f cloudflared", shell=True)
time.sleep(1)

# 1) download cloudflared once
if not os.path.exists("cloudflared"):
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared", shell=True, check=True)

# 2) start Streamlit (CORS/XSRF off so it works behind a tunnel)
subprocess.Popen("streamlit run app.py --server.port 8501 --server.headless true "
                 "--server.enableCORS false --server.enableXsrfProtection false > streamlit_logs.txt 2>&1", shell=True)
time.sleep(6)

# 3) start the tunnel and read the public URL from its log
subprocess.Popen("./cloudflared tunnel --url http://localhost:8501 --no-autoupdate > cloudflared.log 2>&1", shell=True)
url = None
for _ in range(40):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+[.]trycloudflare[.]com", open("cloudflared.log").read())
    if m:
        url = m.group(0); break
print("Open your dashboard here:", url or "tunnel not ready - re-run this cell or check cloudflared.log")

> **If the link does not load:** wait 10–20 seconds and refresh (new tunnels need a moment to register). Check `streamlit_logs.txt` for app errors. As a fallback you can use `pyngrok` with a free ngrok auth token: `!pip install -q pyngrok`, then `ngrok.connect(8501)`.

## 11. Download Deliverables

In [ ]:
from google.colab import files
for f in ["model.joblib", "app.py", "maintenance_rules.py", "data_clean.csv", "metrics.json", "feature_importance.csv"]:
    assert os.path.exists(f), f"missing {f}"
with zipfile.ZipFile("predictive_maintenance_project.zip", "w") as z:
    for f in ["model.joblib", "app.py", "maintenance_rules.py", "data_clean.csv", "metrics.json", "feature_importance.csv"]:
        z.write(f)
files.download("predictive_maintenance_project.zip")

## 12. Summary, Limitations & Future Work

**What was built:** a complete, reproducible workflow — EDA, leakage-safe preprocessing, class-imbalance-aware model comparison, threshold tuning that prioritises recall, documented risk categories, explainability, rule-based maintenance suggestions, a saved model, and a Streamlit dashboard.

**Limitations**
- The data are **synthetic**; real machinery will have noisier, drifting, and differently-distributed data.
- Only ~3% of records are failures, so test-set metrics rest on a small number of positive examples and carry real uncertainty.
- Probabilities are risk scores from a class-weighted model, not calibrated failure rates.
- Maintenance rules are simple thresholds from the dataset documentation, not engineering-validated limits.
- Each record is treated independently; there is no time-series or machine-history information.

**Future work:** failure-type (multi-label) prediction using `TWF/HDF/PWF/OSF/RNF`; additional algorithms (Gradient Boosting, XGBoost); probability calibration; SHAP explanations; equipment-level maintenance history; validation on real sensor data.